<a href="https://colab.research.google.com/github/giuliabugatti09/Movie-Recommendation-System/blob/main/notebooks/Movie_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 Movie Recommendation System: Content-Based Filtering
> **Objective:** Build an AI engine that suggests movies based on plot similarity using Natural Language Processing (NLP).

### 🛠️ Methodology:
1. **Feature Extraction:** Transform movie overviews into numerical vectors using **TF-IDF**.
2. **Similarity Scoring:** Use the **Sigmoid Kernel** to calculate the mathematical "distance" between movies.
3. **Recommendation Engine:** Create a function that maps titles to the top 10 most similar films.

## Environment Setup & Data Loading

In [15]:
!pip install pandas openpyxl
from IPython.display import display
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import sigmoid_kernel

# Load datasets
credits = pd.read_csv("/content/tmdb_5000_credit.csv", encoding='utf-8', on_bad_lines='skip')
movies = pd.read_csv("/content/tmdb_5000_movies.csv", encoding='utf-8', on_bad_lines='skip')

# Merge datasets on 'id'
credits_renamed = credits.rename(columns={"movie_id": "id"})
df = movies.merge(credits_renamed, on='id')

print(f"Dataset Loaded: {df.shape[0]} movies and {df.shape[1]} columns.")
df.head(2)

Dataset Loaded: 4795 movies and 23 columns.


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title_x,vote_average,vote_count,title_y,cast,crew\t
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."


##Data Cleaning & Preprocessing


In [16]:
# Drop unnecessary columns for recommendation
cols_to_drop = ['homepage', 'title_x', 'title_y', 'status', 'production_countries']
df_cleaned = df.drop(columns=cols_to_drop)

# Handle missing values in the 'overview' column (critical for NLP)
df_cleaned['overview'] = df_cleaned['overview'].fillna('')

print("Data Cleaning Complete. Missing values in overview:", df_cleaned['overview'].isnull().sum())

Data Cleaning Complete. Missing values in overview: 0


##NLP Pipeline (TF-IDF Vectorization)


In [17]:
# Initialize TF-IDF Vectorizer
# ngram_range=(1, 3) captures phrases of 1 to 3 words (e.g., "science", "science fiction")
tfv = TfidfVectorizer(min_df=3,
                      max_features=None,
                      strip_accents='unicode',
                      analyzer='word',
                      token_pattern=r'\w{1,}',
                      ngram_range=(1, 3),
                      stop_words='english')

# Transforming 'overview' text into a sparse matrix
tfv_matrix = tfv.fit_transform(df_cleaned['overview'])

print(f"TF-IDF Matrix Shape: {tfv_matrix.shape}")
print("Sparse matrix generated successfully.")

TF-IDF Matrix Shape: (4795, 10400)
Sparse matrix generated successfully.


##Similarity Computation (The Engine)

In [18]:
# Compute the Sigmoid Kernel
# This calculates the similarity score (0 to 1) between every movie pair
sig = sigmoid_kernel(tfv_matrix, tfv_matrix)

# Create a reverse mapping of movie titles to indices
# This allows us to find the index of a movie by its name
indices = pd.Series(df_cleaned.index, index=df_cleaned['original_title']).drop_duplicates()

print("Similarity matrix (Sigmoid Kernel) computed.")

Similarity matrix (Sigmoid Kernel) computed.


## Recommendation Function Logic

In [19]:
def give_recommendations(title, sig_matrix=sig):
    # 1. Check if the movie exists in our index
    if title not in indices:
        return f"Error: '{title}' not found in the database."

    # 2. Get the index of the movie
    idx = indices[title]

    # 3. Get pairwise similarity scores for all movies with that movie
    # enumerate() keeps track of the original movie index
    sig_scores = list(enumerate(sig_matrix[idx]))

    # 4. Sort the movies based on similarity scores in descending order
    sig_scores = sorted(sig_scores, key=lambda x: x[1], reverse=True)

    # 5. Take the top 10 most similar movies (excluding the first one, which is itself)
    sig_scores = sig_scores[1:11]

    # 6. Extract the movie indices
    movie_indices = [i[0] for i in sig_scores]

    # 7. Return the top 10 most similar movie titles
    return df_cleaned['original_title'].iloc[movie_indices]

## Testing & Inference

In [20]:
# Testing with different movies
print("--- Recommendations for 'Avatar' ---")
print(give_recommendations('Avatar'))

print("\n--- Recommendations for 'The Dark Knight Rises' ---")
print(give_recommendations('The Dark Knight Rises'))

print("\n--- Recommendations for 'Newlyweds' ---")
print(give_recommendations('Newlyweds'))

--- Recommendations for 'Avatar' ---
1333                Obitaemyy Ostrov
627                       The Matrix
3596                       Apollo 18
2122                    The American
767                        Supernova
522                 Tears of the Sun
149                          Beowulf
305     The Adventures of Pluto Nash
839                         Semi-Pro
934                 The Book of Life
Name: original_title, dtype: object

--- Recommendations for 'The Dark Knight Rises' ---
293                              Batman Forever
63                              The Dark Knight
1351                                     Batman
421                              Batman Returns
2499                                  Slow Burn
117                               Batman Begins
1173                                        JFK
9            Batman v Superman: Dawn of Justice
3846    Batman: The Dark Knight Returns, Part 2
208                              Batman & Robin
Name: original_title, dt

## Conclusion & Strategic Insights

### ✅ Final Results:
* **NLP Power:** The TF-IDF + Sigmoid approach effectively finds movies with similar themes and plot structures.
* **Scalability:** This method is extremely fast for inference once the similarity matrix is computed.
* **Limitation:** As a "Content-Based" system, it relies heavily on the quality of the `overview`. It doesn't yet consider actors, directors, or genres (unless mentioned in the summary).

**Future Work:** Implement a **Knowledge-Based** approach or a **Metadata Soup** (joining cast, crew, and keywords) to improve recommendation accuracy.